# PIPELINE XỬ LÝ TỰ ĐỘNG DỮ LIỆU SÁCH: "Ca Trù Thê Cách, AB 160, Nôm_Y"

Notebook này triển khai quy trình xử lý tự động toàn diện (*End-to-End Automated Pipeline*) chuyển đổi từ file PDF scan thô sang gói dữ liệu chuẩn quy định theo yêu cầu của đồ án.

### 1. Thông tin Sách & Dữ liệu Nguồn
- **Tên sách chuẩn**: `Ca Trù Thê Cách, AB 160, Nôm_Y` (tuyệt đối đồng bộ với tên file PDF gốc)
- **Ký hiệu văn bản**: AB 160 (Viện Nghiên cứu Hán Nôm / Yale University)
- **Tập tin PDF**: `datasets-midterm/pdf/Ca Trù Thê Cách, AB 160, Nôm_Y.pdf` (Tổng số: **45 trang**)
- **Bố cục văn bản**: Song ngữ Nôm - Quốc ngữ chung trang (**Layout T2b**, viết tay trên vở ô ly học sinh)

### 2. Sơ đồ Kiến trúc Pipeline Toàn diện
<p align="center">
  <img src="imgs/pipeline_architecture.png" alt="Pipeline Architecture" width="92%"/>
</p>

In [ ]:
import os
import sys
import shutil
import json
from pathlib import Path

# Thiết lập đường dẫn môi trường
sys.path.append(os.path.abspath("."))
from core.config import BOOKS, OUTPUT_DIR, DATASETS_MIDTERM_DIR
from core.preprocess import DocumentPreprocessor
from core.aligner import SentenceAligner
from core.cleaner import PPOCRLabelFormatter
from core.evaluator import AlignmentEvaluator
from core.exporter import CorpusExporter

BOOK_KEY = "AB_160"
book_info = BOOKS[BOOK_KEY]
title = book_info["title"]
pdf_path = DATASETS_MIDTERM_DIR / "pdf" / book_info["pdf_filename"]
csv_baseline_path = DATASETS_MIDTERM_DIR / "csv" / book_info["csv_filename"]
book_out_dir = OUTPUT_DIR / title

print(f"Sách cấu hình: {title}")
print(f"Tệp PDF nguồn : {pdf_path} (Tồn tại: {pdf_path.exists()})")
print(f"Thư mục đầu ra: {book_out_dir}")

## GIAI ĐOẠN 1: Bóc tách & Kết xuất Toàn bộ 45 Trang PDF (300 DPI)
Theo quy định, toàn bộ các trang của tập tin PDF gốc phải được kết xuất ở độ phân giải 300 DPI nhằm đảm bảo độ sắc nét cho việc trích xuất tọa độ Bounding Box và nhận dạng OCR.

In [ ]:
preprocessor = DocumentPreprocessor(book_info, OUTPUT_DIR)

print(f"Đang kết xuất ảnh 300 DPI từ tệp: {pdf_path.name}...")
extracted_pages = preprocessor.split_and_extract_images(str(pdf_path))
print(f"==> Đã kết xuất thành công {len(extracted_pages)}/45 trang ảnh.")

## GIAI ĐOẠN 2: Phân loại Trang Tự động (`misc`, `blank`, `content`)
Mỗi trang được phân loại tự động vào các nhóm cấu trúc theo quy chuẩn nộp bài:
- **misc**: Trang bìa trước (Trang 1), Mục lục thư viện (Trang 2), Bìa sau (Trang 45).
- **blank**: Các trang giấy trắng không có nội dung.
- **content**: 42 trang nội dung chính văn (Trang 3 đến 44) &mdash; được đồng bộ vào thư mục `ch/` (chữ Hán Nôm) và `vi/` (chữ Quốc ngữ).

In [ ]:
content_images = []
blank_images = []
misc_images = []

for page_num, img_name, img_path in extracted_pages:
    category = preprocessor.classify_page(img_path, page_num, len(extracted_pages))
    if category == "content":
        content_images.append((page_num, img_name, img_path))
        shutil.copy2(str(img_path), str(preprocessor.dir_ch / img_name))
        shutil.copy2(str(img_path), str(preprocessor.dir_vi / img_name))
    elif category == "blank":
        blank_images.append((page_num, img_name, img_path))
        shutil.copy2(str(img_path), str(preprocessor.dir_blank / img_name))
    else:
        misc_images.append((page_num, img_name, img_path))
        shutil.copy2(str(img_path), str(preprocessor.dir_misc / img_name))

# Sao chép PDF gốc vào thư mục output theo đúng tên quy định
shutil.copy2(str(pdf_path), str(book_out_dir / f"{title}.pdf"))

print(f"Kết quả phân loại toàn bộ 45 trang:")
print(f"  - Trang nội dung (ch/ & vi/) : {len(content_images)} trang")
print(f"  - Trang phụ trợ (misc/)       : {len(misc_images)} trang")
print(f"  - Trang trắng (blank/)        : {len(blank_images)} trang")
print(f"  - Tổng kiểm tra bảo toàn      : {len(content_images) + len(misc_images) + len(blank_images)}/45 trang (100%)")

## GIAI ĐOẠN 3: Dóng hàng Song ngữ T2b & Khôi phục Ký tự bằng Từ điển Offline
Hệ thống ứng dụng thuật toán dóng hàng song song kết hợp kho từ điển Unicode Unihan + Nôm Na Việt (27.612 chữ Nôm) để chuẩn hóa chính tả và đối sánh từng cặp câu.

In [ ]:
aligner = SentenceAligner(book_info, csv_baseline_path)
aligned_records = aligner.align_book_data()

print(f"==> Đã dóng hàng thành công {len(aligned_records)} cặp câu Nôm - Quốc ngữ.")
print("Mẫu 3 cặp câu đầu tiên:")
for r in aligned_records[:3]:
    print(f"  [{r['img_id']}] Câu {r['sentence_id']}: Nôm: '{r['SinoNom_Char']}' <==> QN: '{r['ChuQN_txt']}'")

## GIAI ĐOẠN 4: Phân đoạn Dòng & Sinh Nhãn Chuẩn PPOCRLabel (ch/ & vi/)
Hệ thống tự động phát hiện đường lề vở học sinh ($x \approx 260\text{ px}$) để loại bỏ các ghi chú ngoài lề và số trang, bóc tách chính xác cặp dòng song song T2b thành Bounding Box đúng 4 điểm tọa độ.

In [ ]:
ch_labels_by_img = {}
vi_labels_by_img = {}

page_to_records = {}
for r in aligned_records:
    img_id = r["img_id"]
    if img_id not in page_to_records:
        page_to_records[img_id] = []
    page_to_records[img_id].append(r)

for page_num, img_name, img_path in content_images:
    seg_res = preprocessor.segment_t2b_lines(img_path)
    ch_boxes = seg_res["ch"]
    vi_boxes = seg_res["vi"]
    recs_for_page = page_to_records.get(img_name, [])

    ch_annotated = []
    for b_idx, box in enumerate(ch_boxes):
        text = recs_for_page[b_idx]["SinoNom_Char"] if b_idx < len(recs_for_page) else ""
        ch_annotated.append({"transcription": text, "points": box["points"]})
    ch_labels_by_img[img_name] = ch_annotated

    vi_annotated = []
    for b_idx, box in enumerate(vi_boxes):
        text = recs_for_page[b_idx]["ChuQN_txt"] if b_idx < len(recs_for_page) else ""
        vi_annotated.append({"transcription": text, "points": box["points"]})
    vi_labels_by_img[img_name] = vi_annotated

PPOCRLabelFormatter.write_ppocr_dataset(preprocessor.dir_ch, "ch", ch_labels_by_img)
PPOCRLabelFormatter.write_ppocr_dataset(preprocessor.dir_vi, "vi", vi_labels_by_img)

print("Đã xuất dữ liệu nhãn PPOCRLabel chuẩn:")
print(f"  * ch/Label.txt & ch/fileState.txt: {len(ch_labels_by_img)} trang ảnh")
print(f"  * vi/Label.txt & vi/fileState.txt: {len(vi_labels_by_img)} trang ảnh")

## GIAI ĐOẠN 5: Đánh giá Độ tương đồng Ngữ âm $\ge 95\%$ & Gán nhãn Màu Đỏ/Đen
Hệ thống tính toán độ tương đồng tự động theo công thức quy định:
$$\text{Similarity (\%)} = \left(1 - \frac{\text{Total Red Chars}}{\text{Total Chars}}\right) \times 100\% \ge 95\%$$

In [ ]:
evaluator = AlignmentEvaluator()
eval_report = evaluator.evaluate_dataset(aligned_records)

print("=" * 60)
print("KẾT QUẢ ĐÁNH GIÁ CHẤT LƯỢNG NGỮ ÂM:")
print(f"  * Tổng số ký tự Nôm đánh giá : {eval_report['total_chars']}")
print(f"  * Số ký tự khớp chuẩn (Đen)   : {eval_report['total_black_chars']}")
print(f"  * Số ký tự dị biệt/cần xem (Đỏ): {eval_report['total_red_chars']}")
print(f"  * Tỷ lệ tương đồng ngữ âm     : {eval_report['similarity_percentage']:.2f}%")
print(f"  * Yêu cầu quy chế (>= 95%)    : {'ĐẠT CHUẨN [PASSED]' if eval_report['is_passed'] else 'KHÔNG ĐẠT [FAILED]'}")
print("=" * 60)

## GIAI ĐOẠN 6: Xuất Bản Tập tin Sản phẩm Giao nộp (CSV, XLSX tô màu ARGB, PDF)

In [ ]:
out_csv = book_out_dir / f"{title}.csv"
out_xlsx = book_out_dir / f"{title}.xlsx"

CorpusExporter.export_csv(aligned_records, out_csv)
CorpusExporter.export_xlsx(aligned_records, eval_report["row_evaluations"], out_xlsx)

print("Đã xuất các sản phẩm giao nộp:")
print(f"  1. CSV : {out_csv.name} ({out_csv.stat().st_size:,} bytes)")
print(f"  2. XLSX: {out_xlsx.name} ({out_xlsx.stat().st_size:,} bytes, font ARGB Đen/Đỏ)")
print(f"  3. PDF : {title}.pdf")
print("\n===> HOÀN TẤT PIPELINE XỬ LÝ CHO CUỐN: Ca Trù Thê Cách, AB 160, Nôm_Y!")